In [14]:
import yfinance
import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from datetime import datetime, timezone

pd.set_option('display.max_columns', None)

## Check Existing Columns For Load

We'll separate the price/balance sheet data in sources/ that came directly from yfinance and processed/ which combine all the data from sources at each snapshot in time
We'll also need start indexing processed/ by time 

In [50]:
train = pd.read_csv('data/processed/final.csv')
price = pd.read_csv('data/sources/price.csv')
financials = pd.read_csv('data/sources/financials.csv')

C:\Users\wongs\AppData\Local\Temp\ipykernel_962736\406775871.py:1: DtypeWarning: Columns (119) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('data/processed/final.csv')
C:\Users\wongs\AppData\Local\Temp\ipykernel_962736\406775871.py:3: DtypeWarning: Columns (50) have mixed types. Specify dtype option on import or set low_memory=False.
  financials = pd.read_csv('data/sources/financials.csv')


In [5]:
max_date_by_ticker = train.groupby('ticker')['date'].max().reset_index()
report_days_since_max = train.merge(max_date_by_ticker, on=['ticker', 'date'], how='inner')[['ticker', 'days_from_report_quarterly', 'days_from_report_annual']]
max_date_by_ticker = max_date_by_ticker.merge(report_days_since_max, on='ticker').rename(columns={'date': 'date_max'})
max_date_by_ticker['date_max'] = pd.to_datetime(max_date_by_ticker['date_max'], utc=True)

max_date_by_ticker['days_from_report_annual_to_today'] = max_date_by_ticker['days_from_report_annual'] + np.where(max_date_by_ticker['days_from_report_annual'].notna(), (datetime.now(timezone.utc) - max_date_by_ticker['date_max']).dt.days, np.nan)
max_date_by_ticker['days_from_report_quarterly_to_today'] = max_date_by_ticker['days_from_report_quarterly'] + np.where(max_date_by_ticker['days_from_report_quarterly'].notna(), (datetime.now(timezone.utc) - max_date_by_ticker['date_max']).dt.days, np.nan)

max_date_by_ticker['date_max'] + pd.to_timedelta(datetime.now(timezone.utc) - max_date_by_ticker['date_max'], unit='D')

0     2026-01-13 05:08:39.018056+00:00
1     2026-01-13 05:08:39.018056+00:00
2     2026-01-13 05:08:39.018056+00:00
3     2026-01-13 05:08:39.018056+00:00
4     2026-01-13 05:08:39.018056+00:00
                    ...               
546   2026-01-13 05:08:39.018056+00:00
547   2026-01-13 05:08:39.018056+00:00
548   2026-01-13 05:08:39.018056+00:00
549   2026-01-13 05:08:39.018056+00:00
550   2026-01-13 05:08:39.018056+00:00
Name: date_max, Length: 551, dtype: datetime64[ns, UTC]

In [6]:
financials_quarterly = financials.loc[financials['period'] == 'quarterly', ['ticker', 'date']]
financials_quarterly_max = financials_quarterly.groupby('ticker')['date'].max().reset_index().rename(columns={'date': 'date_financials_quarterly_max'})

financials_annual = financials.loc[financials['period'] == 'annual', ['ticker', 'date']]
financials_annual_max = financials_annual.groupby('ticker')['date'].max().reset_index().rename(columns={'date': 'date_financials_annual_max'})

In [7]:
max_date_by_ticker = price.groupby('ticker')['date'].max().reset_index().rename(columns={'date': 'date_price_max'})
max_date_by_ticker = max_date_by_ticker.merge(financials_annual_max, on='ticker', how='left')
max_date_by_ticker = max_date_by_ticker.merge(financials_quarterly_max, on='ticker', how='left')
max_date_by_ticker['date_financials_annual_max'] = pd.to_datetime(max_date_by_ticker['date_financials_annual_max'], utc=True)
max_date_by_ticker['date_financials_quarterly_max'] = pd.to_datetime(max_date_by_ticker['date_financials_quarterly_max'], utc=True)
max_date_by_ticker['days_from_report_annual_to_today'] = np.where(max_date_by_ticker['date_financials_annual_max'].notna(), (datetime.now(timezone.utc) - max_date_by_ticker['date_financials_annual_max']).dt.days, np.nan)
max_date_by_ticker['days_from_report_quarterly_to_today'] = np.where(max_date_by_ticker['date_financials_quarterly_max'].notna(), (datetime.now(timezone.utc) - max_date_by_ticker['date_financials_quarterly_max']).dt.days, np.nan)
max_date_by_ticker

,ticker,date_price_max,date_financials_annual_max,date_financials_quarterly_max,days_from_report_annual_to_today,days_from_report_quarterly_to_today
0,A,2025-12-04 00:00:00-05:00,2024-10-31 00:00:00+00:00,2025-10-31 00:00:00+00:00,439.0,74.0
1,AAL,2026-01-02 00:00:00-05:00,2024-12-31 00:00:00+00:00,2025-09-30 00:00:00+00:00,378.0,105.0
2,AAP,2026-01-02 00:00:00-05:00,2024-12-31 00:00:00+00:00,2025-09-30 00:00:00+00:00,378.0,105.0
3,AAPL,2025-12-04 00:00:00-05:00,2025-09-30 00:00:00+00:00,2025-09-30 00:00:00+00:00,105.0,105.0
4,ABBV,2025-12-04 00:00:00-05:00,2024-12-31 00:00:00+00:00,2025-09-30 00:00:00+00:00,378.0,105.0
...,...,...,...,...,...,...
553,YUM,2025-12-04 00:00:00-05:00,2024-12-31 00:00:00+00:00,2025-09-30 00:00:00+00:00,378.0,105.0
554,ZBH,2025-12-04 00:00:00-05:00,2024-12-31 00:00:00+00:00,2025-09-30 00:00:00+00:00,378.0,105.0
555,ZBRA,2025-12-04 00:00:00-05:00,2024-12-31 00:00:00+00:00,2025-09-30 00:00:00+00:00,378.0,105.0
556,ZION,2026-01-02 00:00:00-05:00,2024-12-31 00:00:00+00:00,2025-09-30 00:00:00+00:00,378.0,105.0


In [8]:
annual_threshold = 365 
quarterly_threshold = 85

In [9]:
import yfinance as yf
import pandas as pd

def fetch_incremental_daily(tickers, last_date_utc: pd.Timestamp):
    # Yahoo end is exclusive-ish; add a buffer day
    start = (last_date_utc.tz_convert("UTC").date())
    end   = (pd.Timestamp.utcnow().date() + pd.Timedelta(days=1))

    px = yf.download(
        tickers=tickers,
        start=str(start),
        end=str(end),
        interval="1d",
        group_by="ticker",
        auto_adjust=True,
        threads=True,
        progress=False
    )
    return px

def fetch_financials_one(ticker: str) -> dict[str, pd.DataFrame]:
    t = yf.Ticker(ticker)

    return {
        "income_annual": t.income_stmt,
        "income_quarterly": t.quarterly_income_stmt,
        "cashflow_annual": t.cash_flow,
        "cashflow_quarterly": t.quarterly_cash_flow,
        "bs_annual": t.balance_sheet,
        "bs_quarterly": t.quarterly_balance_sheet,
    }

In [25]:
max_date_by_ticker['date_price_max'].value_counts()

date_price_max
2025-12-04 00:00:00-05:00    497
2026-01-02 00:00:00-05:00     56
2025-12-03 00:00:00-05:00      3
2025-11-28 00:00:00-05:00      1
2025-11-26 00:00:00-05:00      1
Name: count, dtype: int64

In [ ]:
update_batch = max_date_by_ticker.groupby('date_price_max')['ticker'].apply(list).to_dict()


In [55]:
import math
import pandas as pd

def chunk_list(xs, chunk_size: int):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")
    for i in range(0, len(xs), chunk_size):
        yield xs[i:i + chunk_size]

def fetch_incremental_daily_batched(
    update_batch: dict,              # {last_date_ts: [tickers]}
    chunk_size: int = 200,
):
    """
    Loops over your {date -> tickers} batches, downloads prices in ticker-chunks,
    returns one long dataframe: columns [date, ticker, open, high, low, close, adj close, volume, ...] (lowercased).
    """
    all_chunks = []

    for last_date, tickers in update_batch.items():
        last_date_utc = pd.Timestamp(last_date)

        # ensure list + dedupe while preserving order
        seen = set()
        tickers = [t for t in tickers if not (t in seen or seen.add(t))]

        for tks in chunk_list(tickers, chunk_size):
            px = fetch_incremental_daily(tickers=tks, last_date_utc=last_date_utc)

            if px is None or px.empty:
                continue

            long = (
                px.stack(level=0)
                  .rename_axis(index=["date", "ticker"])
                  .reset_index()
            )
            long.columns = [c.lower() for c in long.columns]
            all_chunks.append(long)

    if not all_chunks:
        return pd.DataFrame(columns=["date", "ticker"])

    out = pd.concat(all_chunks, ignore_index=True)

    # normalize datetime (yf often returns tz-naive dates in index)
    out["date"] = pd.to_datetime(out["date"], utc=True, errors="coerce")
    out = out.dropna(subset=["date", "ticker"]).sort_values(["ticker", "date"])

    return out


In [56]:
supplemental_price_long = fetch_incremental_daily_batched(update_batch, chunk_size=50)


C:\Users\wongs\AppData\Local\Temp\ipykernel_962736\3205517446.py:34: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  px.stack(level=0)
C:\Users\wongs\AppData\Local\Temp\ipykernel_962736\3205517446.py:34: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  px.stack(level=0)
C:\Users\wongs\AppData\Local\Temp\ipykernel_962736\3205517446.py:34: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence

In [59]:
price = pd.concat([price, supplemental_price_long], ignore_index=True)

In [61]:
price.to_csv('data/sources/price.csv', index=False)